# V38 — Descriptive Statistics: Plain Milk Combined with Healthy Beverages

**Purpose:** Produce Table 1 (Panels A–D) and a deck-ready figure for the **same trip input and household inclusion rule as V38 HMM**. The source Parquet is never edited.

**Five modeled categories:** fruits and vegetables, yogurt, cereal, snacks, and **combined beverages** (healthy drinks **or any qualifying milk**). All original milk outcome codes 1 or 2 count as the preferred healthy beverage choice, with healthy-choice priority when milk and less-healthy beverages appear on the same trip.

**Sample:** 2008–2017; V38's minimum **one observed year** per household (not the V30 three-year restriction). Household-year demographics have equal household-year weight. Category choice rates are calculated **conditional on purchasing that category**; these are not dollar shares.

**Input:** `healthy_purchase_data/processed/hmm_input/hmm_estimation_panel.parquet` (restricted; must stay outside GitHub).

**Outputs:** `healthy_purchase_cycle/descriptive_statistics/V38_combined_beverage/tables/` and `/figures/`. These are aggregated, version-specific results, not microdata; review them before committing to GitHub. DuckDB scratch files stay in `healthy_purchase_data/temp/`.

**Run:** Requires `duckdb`, `pandas`, `numpy`, `matplotlib`. Run all cells locally. The notebook does **not** fit the HMM. The input schema and the combined-beverage rule follow the supplied V38 notebook. If a future HMM changes these, update this descriptive notebook together with it.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import duckdb

DATA_ROOT = Path(r"C:\Users\rwen13\Documents\healthy_purchase_data")
REPO_ROOT = Path(r"C:\Users\rwen13\Documents\healthy_purchase_cycle")
INPUT_PATH = DATA_ROOT / "processed" / "hmm_input" / "hmm_estimation_panel.parquet"
OUTPUT_ROOT = REPO_ROOT / "descriptive_statistics" / "V38_combined_beverage"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
TEMP_DIR = DATA_ROOT / "temp" / "descriptive_statistics_v38_duckdb"

# Match V38: no additional three-year household restriction.
MIN_OBSERVED_YEARS_PER_HOUSEHOLD = 1
DUCKDB_MEMORY_LIMIT = "4GB"
DUCKDB_THREADS = 4
YEAR_START, YEAR_END = 2008, 2017

# Order matches the V38 emission categories.
CATEGORIES = ["fv", "yogurt", "cereal", "snack", "beverage"]
DISPLAY_NAMES = {
    "fv": "Fruits & Vegetables",
    "yogurt": "Yogurt",
    "cereal": "Breakfast Cereal",
    "snack": "Snacks",
    "beverage": "Beverages (including milk)",
}
HEALTHY_LABELS = {
    "fv": "Fresh FV",
    "yogurt": "Fat-free yogurt",
    "cereal": "Low-sugar cereal",
    "snack": "Healthy snacks",
    "beverage": "Healthy drinks or any milk",
}
CHOICE_CODES = {c: (1, 2, 3) if c == "cereal" else (1, 2) for c in CATEGORIES}
EVENTS = {
    "event_new_child": "New child",
    "getting_married": "Marriage",
    "getting_divorce_or_separation": "Divorce or separation",
    "moving": "Moving",
    "getting_job": "Getting a job",
    "retirement": "Retirement",
}
DEMOGRAPHICS = ["age", "family_size_numeric", "income10k_imputed", "K_1"]

# V38 validates these spending columns in its five-category model. Original
# milk variables remain necessary because milk spending enters beverages.
V38_SPENDING = {
    "fv": ["fresh_spending", "non_fresh_spending"],
    "yogurt": ["skimmed_yogurt_spending", "low_fat_yogurt_spending", "other_yogurt_spending"],
    "cereal": ["low_sugar_cereal_spending", "medium_sugar_cereal_spending", "high_sugar_cereal_spending"],
    "snack": ["healthy_snack_spending", "unhealthy_snack_spending"],
    "beverage": ["healthy_beverage_spending", "unhealthy_beverage_spending",
                 "skimmed_milk_spending", "low_fat_milk_spending", "other_milk_spending"],
}

for directory in (TABLE_DIR, FIGURE_DIR, TEMP_DIR):
    directory.mkdir(parents=True, exist_ok=True)
if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"HMM estimation input not found: {INPUT_PATH}")
print(f"Input : {INPUT_PATH}")
print(f"Output: {OUTPUT_ROOT}")


## 1. Validate the V38 estimation input and select eligible households

- Use V38's **one-observed-year minimum**. All sample totals refer to trips retained under that rule.
- Verify original `milk_y` and `beverage_y` independently **before** recombining them; a milk purchase must not mask invalid beverage data.
- In a SQL view only, set `beverage_y = 1` if `milk_y ∈ {1,2}` **or** original `beverage_y = 1`; otherwise set it to `2` for original less-healthy beverage purchases, or `0` for no purchase.
- Validate V38's required demographics, events, choice outcomes and spending columns, retaining the original Parquet unchanged.
- Export only aggregated milk/beverage recoding counts, not identifying records.


In [ ]:
def quoted(col):
    """Quote an SQL identifier without changing the user's column name."""
    return '"' + col.replace('"', '""') + '"'


def setup_connection(path):
    """Attach input read-only and build V38's combined-beverage SQL view."""
    con = duckdb.connect()
    con.execute(f"SET threads={DUCKDB_THREADS}")
    con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
    con.execute("SET temp_directory='" + str(TEMP_DIR).replace("'", "''") + "'")
    con.from_parquet(str(path)).create_view("raw_source")
    available = {row[0] for row in con.execute("DESCRIBE raw_source").fetchall()}
    if not {"milk_y", "beverage_y"}.issubset(available):
        raise ValueError("Both original milk_y and beverage_y are required for V38 recoding.")

    # The exact V38 rule: any original milk takes healthy-choice priority.
    # Null/invalid raw codes are rejected BEFORE replacing beverage_y.
    invalid_original = con.execute("""
        SELECT COUNT(*) FROM raw_source
        WHERE milk_y IS NULL OR beverage_y IS NULL
           OR NOT isfinite(TRY_CAST(milk_y AS DOUBLE))
           OR NOT isfinite(TRY_CAST(beverage_y AS DOUBLE))
           OR milk_y NOT IN (0, 1, 2) OR beverage_y NOT IN (0, 1, 2)
    """).fetchone()[0]
    if invalid_original:
        raise ValueError(f"Invalid original milk/beverage codes: {invalid_original:,} trips")
    con.execute("""
        CREATE TEMP VIEW raw AS SELECT * REPLACE (
            CASE WHEN milk_y IN (1,2) OR beverage_y = 1 THEN 1
                 WHEN beverage_y = 2 THEN 2 ELSE 0 END AS beverage_y
        ) FROM raw_source
    """)
    return con


con = setup_connection(INPUT_PATH)
required = [
    "hhid", "panel_year", "trip_id", "purchase_date", "food_spend",
    "initial_fresh_fv_share_2007", "milk_y", "beverage_y",
    *DEMOGRAPHICS, *EVENTS,
    *[f"{c}_y" for c in CATEGORIES],
    *[column for c in CATEGORIES for column in V38_SPENDING[c]],
]
required = list(dict.fromkeys(required))
available = {row[0] for row in con.execute("DESCRIBE raw").fetchall()}
missing = sorted(set(required) - available)
if missing:
    raise ValueError(f"Missing columns required by V38: {missing}")

numeric = [c for c in required if c not in {"hhid", "trip_id", "purchase_date"}]
invalid_checks = [f"{quoted(c)} IS NULL" for c in required]
invalid_checks += [f"NOT isfinite(TRY_CAST({quoted(c)} AS DOUBLE))" for c in numeric]
invalid_checks += [
    f"panel_year NOT BETWEEN {YEAR_START} AND {YEAR_END}",
    "panel_year != floor(panel_year)",
    "TRY_CAST(purchase_date AS DATE) IS NULL",
]
invalid_checks += [f"{quoted(c)} NOT IN (0,1)" for c in ["K_1", *EVENTS]]
for c in CATEGORIES:
    codes = "0,1,2,3" if c == "cereal" else "0,1,2"
    invalid_checks.append(f"{quoted(c + '_y')} NOT IN ({codes})")
invalid_checks += ["family_size_numeric <= 0", "age <= 0", "age > 120", "income10k_imputed < 0"]

input_n, invalid_n = con.execute(
    "SELECT COUNT(*), COUNT(*) FILTER (WHERE " + " OR ".join(invalid_checks) + ") FROM raw"
).fetchone()
if input_n == 0 or invalid_n:
    raise ValueError(f"Input has {input_n:,} trips and {invalid_n:,} invalid rows. Repair upstream.")
low_prior, high_prior = con.execute("""
    SELECT MIN(initial_fresh_fv_share_2007), MAX(initial_fresh_fv_share_2007) FROM raw
""").fetchone()
if not (0 < low_prior < 1) or abs(high_prior - low_prior) > 1e-12:
    raise ValueError("V38 requires one common initial fresh-FV prior strictly between zero and one.")
if con.execute("""
    SELECT 1 FROM raw GROUP BY hhid, panel_year, trip_id
    HAVING COUNT(*) > 1 LIMIT 1
""").fetchone():
    raise ValueError("Duplicate household-year-trip keys.")
controls = [*DEMOGRAPHICS, *EVENTS, "initial_fresh_fv_share_2007"]
within_year_changes = " OR ".join(
    f"MIN({quoted(c)}) != MAX({quoted(c)})" for c in controls
)
if con.execute(
    "SELECT 1 FROM raw GROUP BY hhid, panel_year HAVING " + within_year_changes + " LIMIT 1"
).fetchone():
    raise ValueError("Demographics, events or initial prior vary within a household-year.")

con.execute(f"""
    CREATE TEMP TABLE eligible_households AS
    SELECT hhid FROM raw GROUP BY hhid
    HAVING COUNT(DISTINCT panel_year) >= {MIN_OBSERVED_YEARS_PER_HOUSEHOLD}
""")
con.execute("""
    CREATE TEMP VIEW trips AS
    SELECT r.* FROM raw r SEMI JOIN eligible_households e USING (hhid)
""")
retained_n = con.execute("SELECT COUNT(*) FROM trips").fetchone()[0]
if retained_n == 0:
    raise ValueError("No eligible trips.")

# Aggregate the original milk/beverage joint codes for reproducibility.
merge_audit = con.execute("""
    SELECT s.milk_y AS original_milk_y,
           s.beverage_y AS original_beverage_y,
           CASE WHEN s.milk_y IN (1,2) OR s.beverage_y = 1 THEN 1
                WHEN s.beverage_y = 2 THEN 2 ELSE 0 END AS combined_beverage_y,
           COUNT(*) AS n_trips
    FROM raw_source s SEMI JOIN eligible_households e USING (hhid)
    GROUP BY 1,2,3 ORDER BY 1,2
""").df()
merge_audit.to_csv(TABLE_DIR / "beverage_merge_audit.csv", index=False)
assert int(merge_audit.n_trips.sum()) == retained_n
print(f"Input trips: {input_n:,}; retained trips: {retained_n:,}; excluded: {input_n-retained_n:,}")
print("Original milk/beverage code merge audit:")
print(merge_audit.to_string(index=False))


## 2. Household-year summary and sample characteristics

Demographic means and SDs weight each observed **household-year equally** (not each shopping trip). Years observed per household is household weighted. V38 permits one-year households; they contribute to emissions and descriptive sample counts but **not** to the observed-step transition likelihood. Life-event frequencies use only household-years after each household's first observed year, including observed-year transitions that skip calendar years (as in V38). These are descriptive frequencies, not causal effects.


In [ ]:
annual_fields = ["CAST(hhid AS VARCHAR) AS hhid", "CAST(panel_year AS INTEGER) AS panel_year",
                 "COUNT(*) AS n_trips"]
annual_fields += [f"FIRST({quoted(c)}) AS {quoted(c)}" for c in [*DEMOGRAPHICS, *EVENTS]]

# Materialize ONLY household-year summaries (much smaller than the trip-level input).
annual_sql = "SELECT " + ", ".join(annual_fields) + " FROM trips GROUP BY hhid, panel_year"
hy = con.execute(annual_sql).df().sort_values(["hhid", "panel_year"]).reset_index(drop=True)
hy["is_transition_year"] = hy.groupby("hhid", sort=False).cumcount().gt(0)
by_household = hy.groupby("hhid", sort=False).agg(
    observed_years=("panel_year", "size"), total_trips=("n_trips", "sum")
)
num_households = len(by_household)
num_hy = len(hy)
num_trips = int(hy.n_trips.sum())
assert num_trips == retained_n
assert int(by_household.observed_years.min()) >= MIN_OBSERVED_YEARS_PER_HOUSEHOLD
transition_hy = hy.loc[hy.is_transition_year]

sample_rows = [
    ("Sample period", "2008–2017", "Calendar years in the final estimation input"),
    ("Number of households", num_households, "Unique retained households"),
    ("Household-year observations", num_hy, "Unique retained household-years"),
    ("Shopping trips", num_trips, "Retained estimation trips"),
    ("Years observed per household", by_household.observed_years.mean(), "Household-weighted mean"),
    ("Shopping trips per household-year", hy.n_trips.mean(), "Household-year-weighted mean"),
]
summary_sample = pd.DataFrame(sample_rows, columns=["variable", "value", "definition"])
# Write aggregate CSV inside the V38-specific output directory.
summary_sample.to_csv(TABLE_DIR / "deck_sample_summary.csv", index=False)

# Mean and sample SD of annual covariates, not weights by grocery-shopping frequency.
demo_order = [
    ("Age of household head (years)", "age"),
    ("Household size", "family_size_numeric"),
    ("Household income ($10,000)", "income10k_imputed"),
    ("Children under 18 (indicator)", "K_1"),
]
demo_rows = [(label, float(hy[col].mean()), float(hy[col].std(ddof=1))) for label, col in demo_order]
event_rows = [(label, float(100 * transition_hy[col].mean())) for col, label in EVENTS.items()]
print(f"Households: {num_households:,} | household-years: {num_hy:,} | trips: {num_trips:,}")
print(f"Transition-eligible household-years: {len(transition_hy):,}")

## 3. Conditional category choices from retained shopping trips

- **Category purchase incidence:** number of trips with an eligible category purchase divided by **all retained trips**.
- **Healthy-choice share:** number of category-purchase trips with preferred outcome `1` divided by category-purchase trips. In V38, combined beverage outcome `1` means **healthy beverage OR any ordinary milk**, even if the trip also includes a less-healthy beverage.
- **Cereal:** `1` = low sugar, `2` = medium without low, `3` = high without low or medium.
- These are **gated trip-level choice frequencies**, not within-category expenditure shares. Milk is not a sixth category.


In [ ]:
category_expressions = ["COUNT(*) AS all_trips"]
for cat in CATEGORIES:
    col = quoted(cat + "_y")
    category_expressions.append(f"COUNT(*) FILTER (WHERE {col} > 0) AS {cat}_purchase")
    for choice in CHOICE_CODES[cat]:
        category_expressions.append(f"COUNT(*) FILTER (WHERE {col} = {choice}) AS {cat}_choice{choice}")
category_counts = con.execute("SELECT " + ", ".join(category_expressions) + " FROM trips").df().iloc[0]
if int(category_counts["all_trips"]) != num_trips:
    raise AssertionError("Trip denominator does not match household-year aggregation")

category_rows = []
for cat in CATEGORIES:
    n_purchase = int(category_counts[f"{cat}_purchase"])
    n_healthy = int(category_counts[f"{cat}_choice1"])
    n_intermediate = int(category_counts[f"{cat}_choice2"]) if cat == "cereal" else None
    n_unhealthy = int(category_counts[f"{cat}_choice3"] if cat == "cereal" else category_counts[f"{cat}_choice2"])
    counts = [n_healthy, n_unhealthy] if cat != "cereal" else [n_healthy, n_intermediate, n_unhealthy]
    assert sum(counts) == n_purchase, f"Category outcomes not exhaustive: {cat}"
    if n_purchase == 0:
        raise ValueError(f"No category purchases in estimation sample: {cat}")
    category_rows.append({
        "category": DISPLAY_NAMES[cat], "category_key": cat,
        "healthy_option": HEALTHY_LABELS[cat],
        "purchase_incidence_pct": 100 * n_purchase / num_trips,
        "healthy_pct": 100 * n_healthy / n_purchase,
        "intermediate_pct": (100 * n_intermediate / n_purchase) if cat == "cereal" else np.nan,
        "unhealthy_pct": 100 * n_unhealthy / n_purchase,
        "category_purchase_trips": n_purchase, "healthy_choice_trips": n_healthy,
        "intermediate_choice_trips": n_intermediate,
        "unhealthy_choice_trips": n_unhealthy,
        "all_estimation_trips": num_trips,
    })
category_df = pd.DataFrame(category_rows)
# Write aggregate CSV inside the V38-specific output directory.
category_df.to_csv(TABLE_DIR / "deck_category_summary.csv", index=False, float_format="%.6f")
print(category_df[["category", "purchase_incidence_pct", "healthy_pct", "intermediate_pct", "unhealthy_pct"]].round(2).to_string(index=False))

## 4. Journal-ready Table 1 (four panels)

Panel A: sample; Panel B: annual demographics; Panel C: five-category purchase behavior; Panel D: life events among **transition-eligible household-years**. Formatted values and underlying numerical columns are saved for manuscript editing. The intermediate cereal column is not applicable to binary categories.


In [ ]:
table_rows = []

def add_row(panel, characteristic, value="", mean=np.nan, sd=np.nan,
            incidence=np.nan, healthy=np.nan, intermediate=np.nan, unhealthy=np.nan):
    table_rows.append({
        "panel": panel, "characteristic": characteristic, "value": value,
        "mean": mean, "sd": sd, "purchase_incidence_pct": incidence,
        "healthy_pct": healthy, "intermediate_pct": intermediate,
        "unhealthy_pct": unhealthy,
    })

for variable, value, _definition in sample_rows:
    text_value = value if isinstance(value, str) else (f"{value:,.0f}" if isinstance(value, (int, np.integer)) else f"{value:,.2f}")
    add_row("Panel A. Sample Characteristics", variable, text_value)

for variable, mean, sd in demo_rows:
    add_row("Panel B. Household Demographics", variable,
            f"{mean:.2f} ({sd:.2f})", mean=mean, sd=sd)

for _, r in category_df.iterrows():
    add_row("Panel C. Category-Level Purchase Behavior", r["category"],
            incidence=r["purchase_incidence_pct"], healthy=r["healthy_pct"],
            intermediate=r["intermediate_pct"], unhealthy=r["unhealthy_pct"])

for label, value in event_rows:
    add_row("Panel D. Life Events", label, f"{value:.2f}%")

journal_table = pd.DataFrame(table_rows)
path_table = TABLE_DIR / "table1_descriptive_statistics.csv"
# Write Table 1 into the V38-specific output directory.
journal_table.to_csv(path_table, index=False, float_format="%.4f")
print(f"Saved: {path_table}")
print("Notes: Panel B unit = household-year; Panel D unit = transition-eligible household-year.")

## 5. Deck-ready descriptive figure

Retain the original one-slide layout: four KPI cards and two horizontal bar panels. Display the **five V38 categories** (including combined milk/beverages); the cereal healthy label remains **Low-sugar cereal**. Save a 300-DPI PNG for slides.


In [ ]:
from matplotlib.patches import FancyBboxPatch, Circle

PALETTE = {"fv": "#26883e", "beverage": "#1877c4", "snack": "#f18c1a",
           "cereal": "#7544af", "yogurt": "#ca4c70"}

fig = plt.figure(figsize=(16, 9), facecolor="#fafafa")
fig.subplots_adjust(0, 0, 1, 1)
background = fig.add_axes([0, 0, 1, 1], zorder=0)
background.axis("off")

def rounded_panel(x, y, w, h, edge="#dfdfdf", fill="#ffffff", linewidth=1.25):
    patch = FancyBboxPatch((x, y), w, h, transform=background.transAxes,
                           boxstyle="round,pad=0.008,rounding_size=0.021",
                           linewidth=linewidth, edgecolor=edge, facecolor=fill)
    background.add_patch(patch)

def kpi_box(bottom, color, number, label, symbol):
    rounded_panel(0.025, bottom, 0.315, 0.205, edge=color, fill="#ffffff", linewidth=1.5)
    background.add_patch(Circle((0.082, bottom + 0.102), 0.055,
                                transform=background.transAxes, facecolor="#f7f9fb",
                                edgecolor=color, linewidth=1.5))
    background.text(0.082, bottom + 0.101, symbol, ha="center", va="center",
                    fontsize=28, color=color, weight="bold", transform=background.transAxes)
    background.text(0.145, bottom + 0.127, number, ha="left", va="center",
                    fontsize=38, weight="bold", color=color, transform=background.transAxes)
    background.text(0.145, bottom + 0.047, label, ha="left", va="center",
                    fontsize=16, color="#202020", transform=background.transAxes)

kpi_box(0.757, "#168546", f"{num_households:,}", "Households", "H")
kpi_box(0.521, "#1877c4", f"{num_hy:,}", "Household-years", "Y")
kpi_box(0.285, "#7544af", f"{num_trips/1_000_000:.2f}M", "Shopping trips", "T")
kpi_box(0.049, "#f09a12", f"{num_trips/num_households:,.1f}", "Mean trips per household", "#")

rounded_panel(0.373, 0.52, 0.607, 0.443)
rounded_panel(0.373, 0.049, 0.607, 0.443)

positions = [(0.47, "Category purchase incidence", "purchase_incidence_pct", "category"),
             (0.091, "Healthy-option share within category trips", "healthy_pct", "healthy_option")]
# Show five V38 emission categories in both right-hand panels.
for top, title, metric, label_col in positions:
    axes = fig.add_axes([0.57, top + 0.047, 0.355, 0.285], zorder=2)
    vals = category_df[metric].to_numpy(float)
    labels = category_df[label_col].tolist()
    keys = category_df["category_key"].tolist()
    y = np.arange(len(keys))
    bars = axes.barh(y, vals, color=[PALETTE[k] for k in keys], height=0.57)
    axes.set_yticks(y, labels=labels, fontsize=10)
    axes.invert_yaxis()
    axes.set_xlim(0, max(100, np.max(vals) + 13))
    axes.set_xticks([0, 20, 40, 60, 80, 100])
    axes.xaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
    axes.grid(axis="x", linestyle="--", color="#dddddd", linewidth=0.8)
    axes.set_axisbelow(True)
    for spine in ["top", "right", "left"]:
        axes.spines[spine].set_visible(False)
    axes.spines["bottom"].set_color("#999999")
    axes.tick_params(axis="both", length=0, pad=7)
    for bar, val in zip(bars, vals):
        axes.text(val + 1.2, bar.get_y() + bar.get_height()/2,
                  f"{val:.1f}%", va="center", ha="left", fontsize=10, weight="bold")
    background.text(0.397, top + 0.355, title, fontsize=16, weight="medium",
                    color="#222222", transform=background.transAxes)

background.text(0.375, 0.012, "2008–2017  |  Five-category V38 HMM sample (milk in beverages)  |  Conditional shares use category-purchase trips",
                fontsize=10, color="#777777", transform=background.transAxes)
figure_path = FIGURE_DIR / "deck_descriptive_figure.png"
# Write the figure into the V38-specific output directory.
fig.savefig(figure_path, dpi=300, facecolor=fig.get_facecolor(), bbox_inches="tight", pad_inches=0.18)
plt.show()
print(f"Saved: {figure_path}")

## 6. Final audit and interpretation notes

**Denominators and V38 alignment**

1. **Panel A:** eligible households, household-years and shopping trips from the same input Parquet and **minimum-one-observed-year rule** as V38. A separate upstream sample filter cannot be reversed here.
2. **Panel B:** mean and sample SD across observed household-years; `K_1` is the under-18-children indicator.
3. **Panel C:** category purchase incidence divides by all retained trips; healthy/intermediate/unhealthy choice percentages divide by **category-purchase trips**. These are **not** food-spending or category-wallet shares.
4. **Panel D:** current-year event frequency among observed household-years **after the first year of each household**. V38 treats gaps as observed-step transitions; the event frequencies reflect that convention, not an event study's estimand.
5. **Cereal:** purchase incidence is based on classified cereal purchases and can exclude trips containing only unclassified cereal.
6. **Beverage recoding:** original `milk_y ∈ {1,2}` or original `beverage_y = 1` produces combined healthy code `1`; otherwise original `beverage_y = 2` produces `2`; neither produces `0`. The joint-code `beverage_merge_audit.csv` records the exact recoding on retained trips.
7. **Milk is not separately modeled:** original milk spending is included in V38's beverage category. The deck figure and Panel C show **five categories**, not six. A trip containing both milk and soft drinks is classified as healthy by the gated rule; this is **not** equivalent to counting only healthy expenditure.

**Data governance:** The restricted Parquet and DuckDB temporary files remain outside GitHub. Aggregate CSV/PNG outputs are versioned in a dedicated folder and should be checked against the project's data-sharing permissions before publication.
